# Exploratory Data Analysis - Revenue Prediction

This notebook explores a sample of the training data using the project's DataLoader.

## Objectives
1. Load sample data efficiently using Dask
2. Explore basic statistics and distributions
3. Identify dense vs sparse features
4. Visualize key relationships with revenue
5. Define feature sets for teacher/student models

## Key Practices
- Uses `DataLoader` from `src/data/loader.py`
- Follows cursorrules: type hints, assertions, clear structure
- Loads small sample for quick iteration
- Avoids repeated `.compute()` calls (Dask best practice)


## 1. Setup and Imports


In [2]:
from __future__ import annotations

import sys
from pathlib import Path
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.loader import DataLoader

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

print("✅ Imports loaded successfully")
print(f"📂 Project root: {project_root}")


✅ Imports loaded successfully
📂 Project root: /home/bigweld/Repos/FME-UPC-datathon-2025


## 2. Configuration

Create a minimal config for DataLoader. We'll load a small time window for exploration.


In [3]:
# Configuration for DataLoader
config = {
    "data": {
        "train_path": str(project_root / "data/raw/train/train"),
        "test_path": str(project_root / "data/raw/test/test"),
        # Load just first 3 datetime partitions for quick exploration
        "train_start": "2025-10-01-00-00",
        "train_end": "2025-10-01-02-00",
        "val_start": "2025-10-01-03-00",
        "val_end": "2025-10-01-04-00",
    },
    "dask": {
        "client": {
            "enabled": False,  # For notebooks, single-threaded is often simpler
        },
        "materialization": {
            "train": {
                "sample_frac": None,  # Load all (already filtered to 3 hours)
                "persist": False,
            },
        },
    },
}

print("✅ Configuration created")
print(f"Training data path: {config['data']['train_path']}")
print(f"Time window: {config['data']['train_start']} to {config['data']['train_end']}")


✅ Configuration created
Training data path: /home/bigweld/Repos/FME-UPC-datathon-2025/data/raw/train/train
Time window: 2025-10-01-00-00 to 2025-10-01-02-00


## 3. Load Sample Data

Use DataLoader to load a small sample efficiently with partition pruning.


In [ ]:
# Initialize DataLoader
loader = DataLoader(config)

# Load train data (filtered to 3-hour window)
print("🔄 Loading training data...")
train_ddf, _ = loader.load_train(validation_split=False)

# Materialize to pandas for exploration
print("🔄 Materializing to pandas (may take a moment)...")
df_sample = loader.materialize(train_ddf, split="train")

print(f"\n✅ Sample loaded: {df_sample.shape[0]:,} rows × {df_sample.shape[1]} columns")
print(f"Memory usage: {df_sample.memory_usage(deep=True).sum() / 1e9:.2f} GB")


2025-11-15 20:50:44 - src.data.loader - INFO - Reading parquet from /home/bigweld/Repos/FME-UPC-datathon-2025/data/raw/train/train with filters=None


🔄 Loading training data...


2025-11-15 20:58:58 - src.data.loader - INFO - ✅ Partition size looks good: 380.4 MB avg
2025-11-15 20:58:58 - src.data.loader - INFO - Full train partitions: 144
2025-11-15 20:58:58 - src.data.loader - INFO - Computing train split to pandas (this may take a while)...


🔄 Materializing to pandas (may take a moment)...


## 4. Initial Data Inspection


In [ ]:
print("=== DATASET SHAPE ===")
print(f"Rows: {df_sample.shape[0]:,}")
print(f"Columns: {df_sample.shape[1]}")

print("\n=== COLUMN NAMES ===")
print(df_sample.columns.tolist())

print("\n=== DATA TYPES ===")
print(df_sample.dtypes)

print("\n=== MISSING VALUES (Top 20) ===")
missing_pct = df_sample.isna().mean().sort_values(ascending=False)
print(missing_pct.head(20))

print("\n=== BASIC STATISTICS ===")
print(df_sample.describe())


## 5. Data Cleaning - Feature Classification

Classify features into:
- **Completely missing** (100% null)
- **Dense features** (low missingness, suitable for student model)
- **Sparse features** (>90% missing, teacher model only)


In [ ]:
from typing import Sequence

# Make a working copy
df = df_sample.copy()

print("=== INITIAL SHAPE ===")
print(f"Shape: {df.shape}")

# 1. Identify completely useless columns (100% missing)
cols_all_missing: Sequence[str] = df.columns[df.isna().mean() == 1.0].tolist()
assert isinstance(cols_all_missing, list), "cols_all_missing must be a list"

print("\n=== COLUMNS WITH 100% MISSING ===")
if len(cols_all_missing) > 0:
    print(cols_all_missing)
    df = df.drop(columns=cols_all_missing)
else:
    print("None found")

# 2. Define dense features (low missingness, good for student model)
candidate_dense_features = [
    "hour", "weekday", "avg_act_days", "avg_daily_sessions",
    "avg_duration", "weekend_ratio", "wifi_ratio",
    "weeks_since_first_seen", "release_msrp", "release_date"
]
dense_features = [col for col in candidate_dense_features if col in df.columns]

print("\n=== DENSE FEATURES (Low Missingness, Good for Student Model) ===")
print(f"Found {len(dense_features)} dense features:")
for feat in dense_features:
    missing_rate = df[feat].isna().mean()
    print(f"  - {feat}: {missing_rate:.1%} missing")

# 3. Identify sparse features (>90% missing)
sparse_threshold = 0.90
sparse_features = df.columns[df.isna().mean() > sparse_threshold].tolist()
assert 0.0 <= sparse_threshold <= 1.0, "sparse_threshold must be between 0 and 1"

print(f"\n=== SPARSE FEATURES (>{sparse_threshold:.0%} Missing) ===")
print(f"Found {len(sparse_features)} sparse features:")
for feat in sparse_features[:10]:  # Show first 10
    missing_rate = df[feat].isna().mean()
    print(f"  - {feat}: {missing_rate:.1%} missing")
if len(sparse_features) > 10:
    print(f"  ... and {len(sparse_features) - 10} more")

# 4. Define feature sets for modeling
# Teacher: all features
teacher_features = df.columns.tolist()

# Student: dense features + key categorical features
candidate_categorical = ["country", "dev_os", "advertiser_category"]
student_features = dense_features + [col for col in candidate_categorical if col in df.columns]

print("\n=== FEATURE SETS FOR MODELING ===")
print(f"Teacher features: {len(teacher_features)} (all available)")
print(f"Student features: {len(student_features)} (dense + key categoricals)")
print(f"\nStudent feature list: {student_features}")


## 6. Exploratory Data Analysis - Revenue Target

Analyze the target variables: `buyer_d7` and `iap_revenue_d7`.


In [ ]:
# Key target variables
TARGET_CLASSIFICATION = "buyer_d7"
TARGET_REGRESSION = "iap_revenue_d7"

assert TARGET_CLASSIFICATION in df.columns, f"{TARGET_CLASSIFICATION} not found in data"
assert TARGET_REGRESSION in df.columns, f"{TARGET_REGRESSION} not found in data"

# Calculate key statistics
buyer_rate = df[TARGET_CLASSIFICATION].mean()
revenue_stats = df[TARGET_REGRESSION].describe()

print("=== TARGET VARIABLE STATISTICS ===")
print(f"\n{TARGET_CLASSIFICATION}:")
print(f"  Buyer rate: {buyer_rate:.2%}")
print(f"  Non-buyers: {(1 - buyer_rate):.2%}")

print(f"\n{TARGET_REGRESSION}:")
print(revenue_stats)

# Create log-transformed revenue for visualization
df["revenue_d7_log1p"] = np.log1p(df[TARGET_REGRESSION])

print("\n✅ Created 'revenue_d7_log1p' column for visualization")


### 6.1 Distribution of Buyers vs Non-Buyers


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x=TARGET_CLASSIFICATION, data=df, ax=ax)
ax.set_title(f"Distribution of {TARGET_CLASSIFICATION}")
ax.set_xlabel("Buyer (1) vs Non-Buyer (0)")
ax.set_ylabel("Count")

# Add percentage labels
for container in ax.containers:
    ax.bar_label(container, fmt='%d')

plt.tight_layout()
plt.show()

print(f"Buyer rate: {buyer_rate:.2%}")
print("Note: Highly imbalanced dataset - consider this for modeling!")


### 6.2 Revenue Distribution (Linear Scale)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df[TARGET_REGRESSION], bins=100, kde=False, ax=ax)
ax.set_title(f"Distribution of {TARGET_REGRESSION} (Linear Scale)")
ax.set_xlabel("Revenue (USD)")
ax.set_xlim(0, 500)  # Focus on 0-500 range to see pattern
plt.tight_layout()
plt.show()

print("Most users have $0 revenue (non-buyers)")


### 6.3 Revenue Distribution (Log Scale)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df["revenue_d7_log1p"], bins=80, kde=False, ax=ax)
ax.set_title("Distribution of Revenue (log1p transformed)")
ax.set_xlabel("log1p(Revenue)")
plt.tight_layout()
plt.show()

print("Log transformation reveals distribution among buyers")


### 6.4 Revenue: Buyers vs Non-Buyers


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(x=TARGET_CLASSIFICATION, y="revenue_d7_log1p", data=df, ax=ax)
ax.set_title("Revenue by Buyer Status (log scale)")
ax.set_xlabel("Buyer Status")
ax.set_ylabel("log1p(Revenue)")
plt.tight_layout()
plt.show()

print("Clear separation between buyers and non-buyers")


## 7. Feature Analysis

Explore relationships between features and revenue.


### 7.1 Revenue by Country (Top 10)


In [ ]:
if "country" in df.columns:
    top_n_countries = 10
    top_countries = df["country"].value_counts().index[:top_n_countries]
    df_top_countries = df[df["country"].isin(top_countries)]
    
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.boxplot(x="country", y="revenue_d7_log1p", data=df_top_countries, ax=ax)
    ax.set_title(f"Revenue by Country (Top {top_n_countries})")
    ax.set_xlabel("Country")
    ax.set_ylabel("log1p(Revenue)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    print(f"Top {top_n_countries} countries by volume:")
    print(df["country"].value_counts().head(top_n_countries))
else:
    print("'country' column not found in dataset")


### 7.2 Revenue by Hour of Day


In [ ]:
if "hour" in df.columns:
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.lineplot(x="hour", y="revenue_d7_log1p", data=df, ax=ax, ci=95)
    ax.set_title("Average Revenue by Hour of Day")
    ax.set_xlabel("Hour of Day")
    ax.set_ylabel("Average log1p(Revenue)")
    ax.set_xticks(range(0, 24, 2))
    plt.tight_layout()
    plt.show()
    
    print("Revenue may vary by time of day - useful feature!")
else:
    print("'hour' column not found in dataset")


### 7.3 Revenue by Advertiser Category (Top 10)


In [ ]:
if "advertiser_category" in df.columns:
    top_n_categories = 10
    top_categories = df["advertiser_category"].value_counts().index[:top_n_categories]
    df_top_categories = df[df["advertiser_category"].isin(top_categories)]
    
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.boxplot(x="advertiser_category", y="revenue_d7_log1p", data=df_top_categories, ax=ax)
    ax.set_title(f"Revenue by Advertiser Category (Top {top_n_categories})")
    ax.set_xlabel("Category")
    ax.set_ylabel("log1p(Revenue)")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    print(f"\nTop {top_n_categories} categories by volume:")
    print(df["advertiser_category"].value_counts().head(top_n_categories))
else:
    print("'advertiser_category' column not found in dataset")


### 7.4 Correlation Heatmap (Dense Numeric Features)


In [ ]:
# Select numeric columns for correlation
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Filter out columns that might have list/array values
clean_numeric_cols: list[str] = []
for col in numeric_cols:
    try:
        # Quick check if column can be used for correlation
        if not df[col].apply(lambda x: isinstance(x, (list, np.ndarray))).any():
            clean_numeric_cols.append(col)
    except Exception:
        continue

assert len(clean_numeric_cols) > 0, "No valid numeric columns found"

# Compute correlation matrix
corr_matrix = df[clean_numeric_cols].corr()

# Plot heatmap
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=False, cmap="coolwarm", center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Correlation Matrix - Numeric Features")
plt.tight_layout()
plt.show()

print(f"\nCorrelation matrix computed for {len(clean_numeric_cols)} numeric features")

# Show top correlations with revenue
if TARGET_REGRESSION in clean_numeric_cols:
    revenue_corr = corr_matrix[TARGET_REGRESSION].abs().sort_values(ascending=False)
    print(f"\nTop 10 features correlated with {TARGET_REGRESSION}:")
    print(revenue_corr.head(11))  # 11 to include revenue itself


## 8. Summary and Next Steps

### Key Findings

1. **Highly Imbalanced Target**: Only ~4% of users are buyers
2. **Sparse Features**: Many features have >90% missing values
3. **Dense Features Identified**: Good candidates for lightweight student model
4. **Categorical Features**: Country, OS, and category show variation in revenue

### Proposed Modeling Strategy

**Teacher Model (Complex):**
- Use all available features (including sparse ones)
- Can handle missing values with advanced techniques
- Goal: Maximum predictive accuracy

**Student Model (Lightweight):**
- Use only dense features + key categoricals
- Fast inference, suitable for real-time deployment
- Learn from teacher's predictions (knowledge distillation)

### Next Steps

1. Feature engineering on dense features
2. Handle categorical encoding
3. Train baseline models
4. Implement teacher-student distillation
5. Evaluate on validation set


In [ ]:
# Clean up
loader.close()
print("✅ DataLoader closed")
print("\n🎉 Analysis complete!")
